In [ ]:
import torch as tc
import igraph as ig
import numpy as np
from matplotlib import pyplot as plt

import sys, os
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '../..'))
sys.path.append(parent_dir)

from src.datasets import NetworksDataset
from src.automata import LLNA
from src.simulation import *

%load_ext autoreload
%autoreload 2

In [ ]:
automaton = LLNA(resolution=2, x=[1], y=[0])

## Simulating with iGraph network models

In [ ]:
def graph2tensor(G, inits=1):
    G = G.as_directed()
    N = len(G.vs)
    E = tc.tensor([ e.tuple for e in G.es ]).T
    h0 = tc.tensor(np.random.randint(0, 2, (inits, N), int))
    return (E, h0)

In [ ]:
graphs = [
    ig.Graph.Erdos_Renyi(n=200, m=4),
    ig.Graph.Watts_Strogatz(dim=1, size=200, nei=2, p=0.1),
    ig.Graph.Barabasi(n=200, m=4, directed=False, power=1),
]

In [ ]:
fig, axes = plt.subplots(len(graphs), 1, sharex=True)
for G, ax, nm in zip(graphs, axes, ['ER', 'WS', 'BA']):
    E, h0 = graph2tensor(G)
    H = automaton(E, h0, T=50)
    ax.imshow(H[0], cmap='gray')
    ax.set(title=f'{nm} model')
plt.tight_layout()
plt.show()

# Using a dataset of pre-generated networks

In [ ]:
DATA_PATH = '../../data/graphs'

In [ ]:
dataset = NetworksDataset(path=DATA_PATH, cached=True, transform=stack_disturbs)

- `path`: where the data is stored (must be generated first)
- `cached`: controls if the data will be kept on memory (True) or loaded from disk every time (False)
- `as_undirected`: if True, force the graph to be treated as undirected, otherwise keep the original format

In [ ]:
dataset.is_directed()

In [ ]:
E, h0 = dataset[10]
print(len(dataset), E.shape, h0.shape)

In [ ]:
automaton = LLNA(resolution=2, x=[1], y=[0])
for  filename in dataset._files[:10]:
    E, h0 = dataset._load(filename)

    H0 = prepare_shape(h0)
    Ht = restore_shape(automaton(E, H0, T=50))

    RES = 2
    RULE = 'B1_S0'
    PATH = f"data/teps/R={RES}/{RULE}/"

    tep = Ht
    teps_defect = []
    defect_nodes = [0, 50]

    for defect_node in defect_nodes:
        tep_original = tep[0,0,:,:]
        tep_disturbed = tep[0,defect_node+1,:,:]
        tep_defect = abs(tep_original-tep_disturbed)
        teps_defect += [tep_defect]

    # default defect TEP for various affected nodes
    fig, axs = plt.subplots(1,2,figsize=(9,3), sharex=True, sharey=True)

    axs[0].imshow(teps_defect[0], cmap='Greys')
    axs[1].imshow(teps_defect[1], cmap='Greys')

    axs[0].set_title(f"Original defect: node {defect_nodes[0]}")
    axs[1].set_title(f"Original defect: node {defect_nodes[1]}")

    fig.suptitle(f'Defect TEPs for various initial defects (rule {RULE})')
    axs[0].set_ylabel(f'$\leftarrow$ Time')
    axs[1].set_ylabel(f'$\leftarrow$ Time')
    # plt.xlabel(f'Nodes (default order)')